# Setup

In [1]:
import os
import torch
from datasets import Dataset
from fontTools.misc.cython import returns
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Use GPU if available, otherwise fall back to CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
torch.set_num_threads(max(1, os.cpu_count()//2))
print(f"Using device: {device}")

C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Step 1. Load the tokenizer and base model
The model HuggingFaceTB/SmolLM2-135M-Instruct is a small, instruction-tuned model that's suitable for this exercise. It has 135 million parameters, making it lightweight and efficient for fine-tuning. It's not the most powerful model, but it's a good choice for demonstrating the concepts of SFT and PEFT with LoRA, especially on a CPU or limited GPU resources.

In [2]:
# Model ID for SmolLM2-135M-Instruct
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to(device)
print(f"model loaded: {model_id}")
print(f"Number of parameters {sum(param.numel() for param in model.parameters() )}")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5586.74it/s]


model loaded: HuggingFaceTB/SmolLM2-135M-Instruct
Number of parameters 134515008


Step 2. Create the dataset

In [3]:
def generate_records():
    for start in range(1, 5):
        for end in range(start + 5, start +8):
            yield {
                # The prompt that is sent to the model
                "prompt": (
                    f"You are a counting assistant. Count from {start} to {end} by 1. Begin: "
                ),
                # Extra values sent to the reward functions
                "start": start,
                "end": end,
            }

ds = Dataset.from_generator(generate_records)
ds[0]

{'prompt': 'You are a counting assistant. Count from 1 to 6 by 1. Begin: ',
 'start': 1,
 'end': 6}

Step 3. Evaluate the base model
Before we fine-tune the model, let's see how it performs on the spelling task. We'll create a helper function to generate a spelling for a given word and compare it to the correct answer.

In [4]:
# Create a helper function that will help us visualize the performance of the model
def check_counting(model, tokenizer, prompt:str, start:int, end:int, max_new_tokens: int=30)->(str,str):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=False) # No parameters = greedy search
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Create the actual counting string
    actual_counting = ", ".join(str(i) for i in range(start, end + 1))+"."
    # Extract the generated counting from the full output string
    proposed_counting = generated_text.split("Begin:")[-1].strip().split("\n")[0].strip()
     # strip any whitespace from the actual counting
    actual_counting = actual_counting.strip()
    print(
        f"Proposed: {proposed_counting} | Actual: {actual_counting} "
        f"| Matches: {'✅' if proposed_counting == actual_counting else '❌'}"
    )

    # Calculate the proportion of the counting that was correct
    num_correct = sum(1 for a, b in zip(actual_counting, proposed_counting) if a == b)

    return num_correct / len(actual_counting)  # Return proportion correct

In [5]:
# Evaluate the base model's counting ability
proportion_correct = 0.0
for example in ds:
    prompt = example["prompt"]
    start = example["start"]
    end = example["end"]
    result = check_counting(model, tokenizer, prompt, start, end)
    proportion_correct += result
print(f"{proportion_correct}/{len(ds)} sequences correct")

Proposed: 1, 2, 3, 4, 5, 6. | Actual: 1, 2, 3, 4, 5, 6. | Matches: ✅
Proposed: 1, 2, 3, 4, 5, 6, 7. | Actual: 1, 2, 3, 4, 5, 6, 7. | Matches: ✅
Proposed: 1, 2, 3, 4, 5, 6, 7, 8. | Actual: 1, 2, 3, 4, 5, 6, 7, 8. | Matches: ✅
Proposed: 1, 2, 3, 4, 5, 6, 7. | Actual: 2, 3, 4, 5, 6, 7. | Matches: ❌
Proposed: 2, 4, 6, 8, 10, 12, 14, 16, 18 | Actual: 2, 3, 4, 5, 6, 7, 8. | Matches: ❌
Proposed: 1, 2, 3, 4, 5, 6, 7, 8, 9. | Actual: 2, 3, 4, 5, 6, 7, 8, 9. | Matches: ❌
Proposed: 3, 4, 5, 6, 7, 8. | Actual: 3, 4, 5, 6, 7, 8. | Matches: ✅
Proposed: 3, 4, 5, 6, 7, 8, 9. | Actual: 3, 4, 5, 6, 7, 8, 9. | Matches: ✅
Proposed: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. | Actual: 3, 4, 5, 6, 7, 8, 9, 10. | Matches: ❌
Proposed: 4, 5, 6, 7, 8, 9. | Actual: 4, 5, 6, 7, 8, 9. | Matches: ✅
Proposed: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10. | Actual: 4, 5, 6, 7, 8, 9, 10. | Matches: ❌
Proposed: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, | Actual: 4, 5, 6, 7, 8, 9, 10, 11. | Matches: ❌
9.281692851053466/12 sequences correct


Step 4. Configure LoRA and train the model
Let’s attach a LoRA adapter to the base model. We use a LoRA config so only a tiny fraction of parameters are trainable. Read more here: LoRA.

In [6]:
# Print how many params are trainable at first
trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
total = sum(param.numel() for param in model.parameters())
print(
    f"Trainable params BEFORE: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)
lora_config = LoraConfig(
    r=64, # Rank of the update matrices. Lower is fewer parameters.
    lora_alpha=16, # LoRA scaling factor. Generally set to 16.
    lora_dropout=0.05,  # Dropout for LoRA layers
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
# Print the number of trainable parameters after applying LoRA
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params AFTER: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)

Trainable params BEFORE: 134,515,008 / 134,515,008 (100.00%)
Trainable params AFTER: 3,686,400 / 138,201,408 (2.67%)


In [7]:
def reward_starting_at_start(completions: list[str],
                             start: list[str],
                             **kwargs)->list[float]:

    start_list = start

    return [
        1.0 if completion.startswith(str(start)) else 0.0
        for completion, start in zip(completions, start_list)
    ]

assert reward_starting_at_start(
    completions=[
        "1, 2, 3",
        "2, 3, 4",
    ],
    start=[1, 1],
) == [1.0, 0.0]

In [8]:
def reward_using_comma_separated_numbers(completions: list[str],
                                        **kwargs)->list[float]:
    return [
        1.0 if ", " in completion else 0.0 for completion in completions
    ]

assert reward_using_comma_separated_numbers(completions=["1, 2, 3", "2 3 4"]) == [
    1.0,
    0.0,
]

In [9]:
import re
def reward_counting_by_one(completions: list[str], **kwargs)->list[float]:
   rewards = []
   for completion in completions:
     # split on all groups of non-numeric characters
     completion  = re.split(r"[^0-9]+", completion)
     numbers = [int(c) for c in completion if c and c.isdigit()]
     difference =[numbers[i] - numbers[i- 1] for i in range(1, len(numbers))]
     if all(dif == 1 for dif in difference ):
        rewards.append(1.0)
     else:
        rewards.append(0.0)
   return rewards


assert reward_counting_by_one(completions=["1, 2, 3, 4.", "2, 4, 6, 8."]) == [1.0, 0.0]

In [10]:
def reward_ending_at_end(completions: list[str], end: list[str], **kwargs)->list[float]:
    end_list = end
    # Remove any punctuation from the end of the completions
    completions = [completion.rstrip(".,!?") for completion in completions]
    return [
        1.0 if completion.endswith(str(end)) else 0.0
        for completion, end in zip(completions, end_list)
    ]

assert reward_ending_at_end(
    completions=["1, 2, 3, 4.", "2, 4, 6"],
    end=[4, 8],
) == [1.0, 0.0]

In [11]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="data/counting-grpo",
    max_completion_length=30,  # The maximum number of tokens to generate
    logging_steps=5,  # Log every 5 steps
    learning_rate=5e-5,  # The learning rate for the optimizer
    num_train_epochs=10,  # We'll train just for a few epochs
    per_device_train_batch_size=16,  # The batch size for training
    num_generations=8,  # Determines the number of completions to compute for each single prompt
    # per_device_train_batch_size / num_generations determines the number of simultaneous prompts to consider
    lr_scheduler_type="cosine",  # Use a cosine scheduler to reduce the learning rate over time
    beta=0.0,  # beta=0.0 means no KL penalty
    bf16=False
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        reward_starting_at_start,
        reward_using_comma_separated_numbers,
        reward_counting_by_one,
        reward_ending_at_end,
    ],
    args=training_args,
    train_dataset=ds,
)
trainer.train()

C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
5,0.002445


KeyboardInterrupt: 